# 07 — Main pipeline

Собирает итоговый `data/submission_v8.csv`, переиспользуя кэши, созданные предыдущими ноутбуками. 

## в `data/` до запуска должно лежать
- `ocr_cache.json` — кэш PaddleOCR + zxing + QR-декодеров (создаётся `04_ocr.ipynb`)
- `vlm_cache_v2.json`, `vlm_cache_v3.json` — два прогона Qwen2.5-VL-7B на Kaggle T4×2 (создаются `05_vlm_kaggle.ipynb`)
- `best_crops_meta.json`, `per_frame_tracks.json` — метаданные кропов и треков (создаются `03_track_and_extract.ipynb`)
- `crops_meta.json` — GT-разметка (входные данные)

- **parsers_v2** — расширенное pair-окно для цен, force `,99` для int-only fallback, fuzzy `product_name` (RapidFuzz + Latin→Cyrillic fold), мягкая детекция К/Ш, tolerance к OCR-путаницам O→0/l→1
- **Гибридный VLM merge** — v3 primary для `code/id_sku/barcode`, v2 primary для `print_datetime`
- **Blocklist** — ~80 захардкоженных VLM-плейсхолдеров (`4670025474665`, `01.01.2023 12:34` и т.д.) отбрасываются ещё до валидации
- **QR mirror** — `price1_qr` ← `price_default`, `price4_qr` ← `price_card`, `qr_code_barcode` ← `barcode` (по GT эти поля совпадают в 88-99% случаев). Конвертация запятой в точку: GT-конвенция — `price_X` с запятой, `price_X_qr` с точкой.
- **Code dedup** — гасит подозрительно частый `code` в больших видео (>40% треков с одним значением → VLM-галлюцинация)
- **additional_info snap** — fuzzy-приближение к закрытому словарю
- **Product name canonicalization** — обрезка bleed-in после volume anchor (`0.75L`), нормализация пробелов под GT-формат

## Output
- `data/submission_v8.csv`
- `data/eval_report_v8.json` — метрики

## Метрика
- **strict** — символ-в-символ
- **fuzzy** — `product_name` через `fuzz_token_set ≥ 80` (диагностический score, чтобы понять, упираемся ли в OCR-ошибки названия товара)

Финальный результат: **3/157 = 1.9%** (strict) на labeled-видео.

In [13]:
import json, re, sys
from pathlib import Path
from collections import defaultdict, Counter
import nbformat

# ROOT — директория, содержащая data/. Работает и при запуске из solution/notebooks/
# (как для разработки в исходном репо), и когда вся solution/ является корнем.
_cwd = Path.cwd()
ROOT = None
for c in [_cwd, *_cwd.parents][:6]:
    if (c / 'data').is_dir():
        ROOT = c
        break
if ROOT is None:
    ROOT = _cwd

# src/ — где лежит parsers_v2.py. Может быть либо ROOT/src (solution как корень),
# либо ROOT/solution/src (repo_root как корень)
_src_candidates = [ROOT / 'src', ROOT / 'solution' / 'src']
SRC_DIR = next((p for p in _src_candidates if p.is_dir()), _src_candidates[0])
sys.path.insert(0, str(SRC_DIR))

# 06_parsers.ipynb — пробуем разные расположения
_parsers_candidates = [
    _cwd / '06_parsers.ipynb',
    ROOT / 'notebooks' / '06_parsers.ipynb',
    ROOT / 'solution' / 'notebooks' / '06_parsers.ipynb',
]
PARSERS_NB = next((p for p in _parsers_candidates if p.exists()), _parsers_candidates[0])

parsers_nb = nbformat.read(PARSERS_NB, as_version=4)
for cell in parsers_nb.cells:
    if cell.cell_type == 'code':
        exec(cell.source, globals())

import parsers_v2
from parsers_v2 import (
    parse_prices_v2,
    parse_special_symbols_v2,
    parse_discount_amount_v2,
    normalize_product_name,
    product_name_match,
)
# parsers_v2 функциям нужны globals — перепрокидываем
for name in ('OcrLine', 'SUPER_TRANS', '_extract_integer', '_try_inline_price',
             'cx', 'cy', '_SOURCE_PRIORITY', '_fmt_price_ru'):
    setattr(parsers_v2, name, globals()[name])

print(f'ROOT:       {ROOT}')
print(f'src:        {SRC_DIR}')
print(f'parsers:    {PARSERS_NB}')
print('Parsers v1+v2 loaded')

# Загрузка данных
ocr_cache = json.loads((ROOT / 'data' / 'ocr_cache.json').read_text())
meta_all  = json.loads((ROOT / 'data' / 'best_crops_meta.json').read_text())
gt        = json.loads((ROOT / 'data' / 'crops_meta.json').read_text())
per_frame = json.loads((ROOT / 'data' / 'per_frame_tracks.json').read_text())

# Гибрид: v3 для code/id_sku/barcode, v2 для print_datetime
vlm_cache_v3_path = ROOT / 'data' / 'vlm_cache_v3.json'
vlm_cache_v2_path = ROOT / 'data' / 'vlm_cache_v2.json'
vlm_cache_v3 = json.loads(vlm_cache_v3_path.read_text()) if vlm_cache_v3_path.exists() else {}
vlm_cache_v2 = json.loads(vlm_cache_v2_path.read_text()) if vlm_cache_v2_path.exists() else {}

all_keys = set(vlm_cache_v3) | set(vlm_cache_v2)
vlm_cache = {}
for k in all_keys:
    r3 = vlm_cache_v3.get(k, {}) or {}
    r2 = vlm_cache_v2.get(k, {}) or {}
    vlm_cache[k] = {
        'print_datetime': r2.get('print_datetime') or r3.get('print_datetime'),
        'id_sku':         r3.get('id_sku')         or r2.get('id_sku'),
        'barcode':        r3.get('barcode')        or r2.get('barcode'),
        'code':           r3.get('code')           or r2.get('code'),
    }
print(f'OCR cache: {len(ocr_cache)} crops, meta: {len(meta_all)} tracks, GT: {len(gt)}, VLM v2: {len(vlm_cache_v2)}, v3: {len(vlm_cache_v3)}, hybrid: {len(vlm_cache)}')

barcode: 4670025474665
checksum OK: True
id_sku: 270207736530
datetime full : 03.04.2026 3:08
datetime fixed: 03.04.2026 3:08
datetime split: 03.04.2026 3:08
discount: -48%
prices full : {'price_default': None, 'price_card': '129,99'}
prices fallback (int-only default): {'price_default': '252,00', 'price_card': '129,99'}
K в кропе : К
Ш в кропе : Ш
латин. K  : К
в слове   : None
пусто     : None
ROOT:       /Users/alex/ML Projects/Lenta Tech Life Hack/solution
src:        /Users/alex/ML Projects/Lenta Tech Life Hack/solution/src
parsers:    /Users/alex/ML Projects/Lenta Tech Life Hack/solution/notebooks/06_parsers.ipynb
Parsers v1+v2 loaded
OCR cache: 5322 crops, meta: 544 tracks, GT: 157, VLM v2: 544, v3: 544, hybrid: 544


## Aggregate records using v2 parsers (где применимо)

In [14]:
def dict_to_line(d):
    return OcrLine(d['text'], d['x'], d['y'], d['w'], d['h'], d['conf'])

def build_product_name(lines):
    """Из 06: top-half лидеров с буквами как product_name."""
    if not lines: return None
    max_y = max(L.y + L.h for L in lines)
    cutoff = max_y * 0.55
    top = [L for L in lines if (L.y + L.h / 2) < cutoff
                            and any(c.isalpha() for c in L.text)
                            and len(L.text.strip()) >= 2]
    if not top: return None
    top.sort(key=lambda L: (L.y, L.x))
    name = ' '.join(L.text.strip() for L in top)
    return name if len(name) >= 5 else None

# Валидаторы (из 06, нужны для финальной фильтрации)
def is_valid_ean13(s):
    if not s: return False
    d = ''.join(c for c in str(s) if c.isdigit())
    return len(d) == 13 and ean13_checksum_ok(d)

def valid_barcode(s):
    if not s: return False
    d = ''.join(c for c in str(s) if c.isdigit())
    if len(d) == 13: return ean13_checksum_ok(d)
    return len(d) in (8, 12)

def valid_id_sku(s):
    if not s: return False
    d = ''.join(c for c in str(s) if c.isdigit())
    return 9 <= len(d) <= 12

def valid_price(s):
    if not s: return False
    try:
        f = float(str(s).replace(',', '.'))
        return 10.0 <= f <= 99999.0
    except Exception: return False

def valid_datetime(s):
    if not s: return False
    return bool(re.fullmatch(r'\d{2}\.\d{2}\.\d{4}( \d{1,2}:\d{2})?', str(s)))

def valid_discount(s):
    if not s: return False
    m = re.fullmatch(r'-\d{1,2}%', str(s))
    if not m: return False
    return 1 <= int(str(s)[1:-1]) <= 99

def valid_special_sym(s):
    return s in ('К', 'Ш')

def valid_code(s):
    # GT formats: 'NNNN - NNNN' range, 'XX_NNNNNN' single, 'XX_NNNNNN - NNNNNN' range with prefix, '024 017_1_6_2' rare
    if not s: return False
    s = str(s).strip()
    patterns = [
        r'\d{4,7}\s*-\s*\d{4,7}',
        r'\d{2}_\d{4,7}(\s*-\s*\d{4,7})?',
        r'\d{2}_[А-ЯA-Z]{2,5}',
        r'\d+ \d+(_\d+)+',
    ]
    return any(re.fullmatch(p, s) for p in patterns)

VALIDATORS = {
    'barcode': valid_barcode, 'id_sku': valid_id_sku,
    'price_default': valid_price, 'price_card': valid_price,
    'print_datetime': valid_datetime, 'discount_amount': valid_discount,
    'special_symbols': valid_special_sym,
}

CSV_FIELDS = [
    'video', 'track_id',
    'product_name', 'price_default', 'price_card', 'price_discount',
    'barcode', 'discount_amount', 'id_sku', 'print_datetime',
    'code', 'additional_info', 'color', 'special_symbols',
    'qr_code_barcode', 'price1_qr', 'price2_qr', 'price3_qr', 'price4_qr',
    'wholesale_level_1_count', 'wholesale_level_1_price',
    'wholesale_level_2_count', 'wholesale_level_2_price',
    'action_price_qr', 'action_code_qr',
]

records = []

for m in meta_all:
    field_votes = defaultdict(Counter)
    name_votes  = Counter()
    qr_first = {}
    zxing_bc = None
    
    for c in m['crops']:
        cd = ocr_cache.get(c['crop_path'], {})
        lines = [dict_to_line(d) for d in cd.get('lines', [])]
        # 1d barcode
        zx = cd.get('zxing')
        if zx and is_valid_ean13(zx) and zxing_bc is None:
            zxing_bc = zx
        # QR
        qr = cd.get('qr', {}) or cd.get('qr_alt', {})
        if qr and not qr_first:
            qr_first = qr

        if not lines: continue

        # v1-парсеры для большинства полей + v2 для прайсов/special/discount
        v_parsed = {
            'barcode':         parse_barcode(lines),
            'id_sku':          parse_id_sku(lines, parse_barcode(lines)),
            'print_datetime':  parse_print_datetime(lines),
            'discount_amount': parse_discount_amount_v2(lines),    # ← v2
            'additional_info': parse_additional_info(lines),
            'special_symbols': parse_special_symbols_v2(lines),    # ← v2
            **parse_prices_v2(lines),                              # ← v2
        }
        for k, v in v_parsed.items():
            if v: field_votes[k][v] += 1

        pn = build_product_name(lines)
        if pn: name_votes[pn] += 1

    parsed = {k: votes.most_common(1)[0][0] for k, votes in field_votes.items()}
    product_name = name_votes.most_common(1)[0][0] if name_votes else None

    # Финальная запись с валидацией
    rec = {f: 'нет' for f in CSV_FIELDS}
    rec['video'] = m['video']
    rec['track_id'] = m['track_id']
    rec['color'] = 'red'
    rec['price_discount'] = 'нет'
    rec['code'] = 'нет'
    rec['special_symbols'] = 'нет'

    for k in ('barcode', 'id_sku', 'print_datetime', 'discount_amount',
              'additional_info', 'special_symbols',
              'price_card', 'price_default'):
        v = parsed.get(k)
        if not v: continue
        validator = VALIDATORS.get(k)
        if validator and not validator(v): continue
        rec[k] = v

    if zxing_bc: rec['barcode'] = zxing_bc
    for k, v in qr_first.items():
        if k in rec and v is not None:
            rec[k] = v
    if product_name: rec['product_name'] = product_name

    records.append(rec)

print(f'Records: {len(records)}')
filled = {f: sum(1 for r in records if r[f] != 'нет') for f in CSV_FIELDS if f not in ('video','track_id','color')}
print('Coverage (top fields):')
for f, n in sorted(filled.items(), key=lambda x: -x[1])[:10]:
    print(f'  {f:<20} {n}/{len(records)} = {n/len(records):.0%}')

Records: 544
Coverage (top fields):
  price_card           360/544 = 66%
  product_name         234/544 = 43%
  price_default        207/544 = 38%
  discount_amount      202/544 = 37%
  additional_info      69/544 = 13%
  special_symbols      54/544 = 10%
  id_sku               31/544 = 6%
  barcode              26/544 = 5%
  qr_code_barcode      2/544 = 0%
  price1_qr            2/544 = 0%


## VLM merge — гибрид v2+v3 с blocklist

Мерджит VLM-предсказания (`code`, `id_sku`, `barcode`, `print_datetime`) в `records` поверх OCR-результата.

**Политика:**
1. `is_placeholder` → отбрасываем известные галлюцинации (даты 2023 года, barcode `4670025474665`, id_sku `270207736530`, code `01_025019`)
2. `validator` → отбрасываем формально невалидные (неправильная длина, нет EAN-13 checksum, неправильный regex)
3. Записываем только если у OCR в этом поле было `'нет'` — не перезаписываем валидные OCR-значения

In [15]:
# Placeholder blocklist: VLM-hallucinations to reject.
# Values that ALSO appear in GT (rare) are kept blocked when GT count << hallucination count.
VLM_PLACEHOLDERS = {
    'barcode': {
        '4670025474665',  # 1 GT vs 285 hallucinations
        '4000000000000', '4006300000001', '4006360000001', '4006373000000',
        '4006920000000', '4006321000001', '4006372000000', '4006591234567',
        '4006666666666', '4006949000000', '4006912345678',
        '5412345678901', '4820000000000', '4600000000000', '4900000000000',
        '6666666666666', '9999999999999', '1234567890123', '7890123456789',
        '3500000000000', '3800000000000', '6000000000000', '3060000000000',
        '4001234567890', '4678901234567', '4956789012345',
    },
    'id_sku': {
        '270207736530',  # 1 GT vs 100+ hallucinations
        '1234567890', '123456789012', '12345678901234',
        '666666666666', '6666666666', '999999999', '699999999',
        '876543210987', '6543210987', '654321098765', '6789012345',
        '3456789012', '9876543210', '8912345678', '6912345678',
        '65912345678', '3968754321', '39012345678', '4486789012345',
        '6612345678', '760000000000', '696000000000',
        '270062000000', '270800000000', '495678901234', '756890123456',
        '768912345678',
    },
    'code': {
        '025017 - 026015', '000000 - 000000',
        '012345 - 067890', '012345 - 678901',
        '123456 - 789012', '666666 - 666666',
        '111111 - 111111', '222222 - 222222', '282828 - 282828',
        # v3 memorized GT — 83% of cache has this value; 17 GT match vs 433 FP
        '01_025019',
    },
    'print_datetime': {
        # All 2023 dates are hallucinations (GT is 2025-2026)
        '01.01.2023 12:34', '01.05.2023 10:30', '01.05.2023 14:30',
        '01.05.2023 14:35', '01.01.2023 15:30', '01.05.2023 10:34',
        '01.05.2023 14:32', '01.05.2023 12:34', '01.08.2023 15:30',
        '05.06.2023 14:30', '01.01.2023 15:34', '01.05.2023 15:30',
        # v3 cache placeholders (2026 but still hallucinated — not in GT)
        '01.01.2026 12:34', '01.05.2026 10:34',
    },
}

VLM_VALIDATORS = {
    'print_datetime': valid_datetime,
    'id_sku':         valid_id_sku,
    'barcode':        valid_barcode,
    'code':           valid_code,
}

def is_placeholder(field, value):
    return value in VLM_PLACEHOLDERS.get(field, set())

n_filled = Counter()
n_kept = Counter()
n_invalid = Counter()
n_blocked = Counter()

for r in records:
    key = f'{r["video"]}/{r["track_id"]}'
    vlm_rec = vlm_cache.get(key)
    if not vlm_rec: continue
    for f, validator in VLM_VALIDATORS.items():
        v = vlm_rec.get(f)
        if not v: continue
        if is_placeholder(f, v):
            n_blocked[f] += 1
            continue
        if not validator(v):
            n_invalid[f] += 1
            continue
        if r.get(f) in (None, '', 'нет'):
            r[f] = v
            n_filled[f] += 1
        else:
            n_kept[f] += 1

print('VLM merge (hybrid v2+v3, blocklist):')
print(f'  {"FIELD":<18} {"FILLED":>8} {"KEPT":>6} {"INVALID":>9} {"BLOCKED":>9}')
for f in VLM_VALIDATORS:
    print(f'  {f:<18} {n_filled[f]:>8} {n_kept[f]:>6} {n_invalid[f]:>9} {n_blocked[f]:>9}')

VLM merge (hybrid v2+v3, blocklist):
  FIELD                FILLED   KEPT   INVALID   BLOCKED
  print_datetime          101      0         1       441
  id_sku                    6      2       220       315
  barcode                   3      3        12       461
  code                     22      0        10       502


## DB sanity filter — drop barcode/qr_code_barcode не из реальной БД товаров

`db_hack.csv` — реальная БД Lenta (356K товаров, cp1251). 99% GT-`barcode` есть в БД, но из 29 наших OCR/VLM-предсказаний в БД попадает только 1. Остальные 28 — мусор/галлюцинации, прошедшие EAN-13 checksum.

Заменяем такие barcode на `'нет'`. Это не добавляет новых правильных значений, но превращает false positive в правильный match, когда GT-`barcode` для этого ценника = `'нет'`.

In [16]:
import csv

# Загружаем БД Lenta (cp1251, ; разделитель). 356K товаров.
DB_PATH = ROOT / 'data' / 'db_hack.csv'

if not DB_PATH.exists():
    print(f'WARN: {DB_PATH} не найден — DB filter пропущен.')
    db_codes = set()
else:
    db_codes = set()
    with DB_PATH.open(encoding='cp1251') as f:
        rdr = csv.reader(f, delimiter=';')
        next(rdr, None)  # header
        for row in rdr:
            if len(row) == 2:
                db_codes.add(row[1].strip())
    print(f'DB загружена: {len(db_codes)} уникальных кодов')

# Фильтруем barcode: если значение есть, но не в БД → 'нет'.
# qr_code_barcode пока не трогаем — он будет перезаполнен на шаге QR mirror из barcode.
n_dropped = 0
n_kept = 0
n_already_net = 0
for r in records:
    bc = r.get('barcode')
    if bc in (None, '', 'нет'):
        n_already_net += 1
        continue
    if bc in db_codes:
        n_kept += 1
    else:
        r['barcode'] = 'нет'
        n_dropped += 1

print(f'\nDB filter on barcode:')
print(f'  было нет:       {n_already_net}')
print(f'  оставили (в БД): {n_kept}')
print(f'  сбросили в нет:  {n_dropped}')

DB загружена: 356277 уникальных кодов

DB filter on barcode:
  было нет:       515
  оставили (в БД): 1
  сбросили в нет:  28


## price_default validator — алгоритмическая проверка integer-части

Анализом GT нашли: `discount_displayed = floor(100 * (1 - price_card / price_default))` работает в 98% случаев.

Значит из `(price_card, discount_amount)` можем посчитать допустимый ДИАПАЗОН для integer-части `price_default`:
- displayed disc D% → actual ∈ [D, D+1) → price_default ∈ [price_card/(1 - D/100), price_card/(1 - (D+1)/100))

Если наш OCR-распознанный integer выходит за этот диапазон → сбрасываем `price_default` в `'нет'`. Это не добавляет новых правильных значений (точные центы из формулы не выводятся), но убирает явно ошибочные предсказания, превращая false-positive в `'нет'`, которое матчит, когда GT тоже `'нет'`.

In [17]:
# Algorithmic price_default validator
# discount_displayed = floor(100 * (1 - price_card / price_default))
# → price_default integer должен попадать в диапазон [pc/(1-D/100), pc/(1-(D+1)/100))

def _parse_price(s):
    if not s or s == 'нет': return None
    try:
        return float(str(s).replace(',', '.'))
    except Exception:
        return None

def _parse_disc(s):
    if not s or s == 'нет': return None
    try:
        return int(str(s).replace('%', '').replace('-', '').strip())
    except Exception:
        return None

n_dropped_pd = 0
n_kept_pd = 0
n_no_signal = 0
for r in records:
    pd_str = r.get('price_default')
    if pd_str in (None, '', 'нет'):
        continue
    pd_val = _parse_price(pd_str)
    pc_val = _parse_price(r.get('price_card'))
    d_pct  = _parse_disc(r.get('discount_amount'))

    if pd_val is None:
        continue
    if pc_val is None or d_pct is None or d_pct == 0:
        # Нет валидаторской пары — не можем проверить, оставляем как есть
        n_no_signal += 1
        continue

    # Допустимый диапазон integer-части
    low  = pc_val / (1 - d_pct / 100.0)            # минимум (включительно)
    high = pc_val / (1 - (d_pct + 1) / 100.0)      # максимум (исключительно)

    pd_int = int(pd_val)
    # Допуск ±1 на округление
    if int(low) - 1 <= pd_int < int(high) + 1:
        n_kept_pd += 1
    else:
        r['price_default'] = 'нет'
        n_dropped_pd += 1

print(f'price_default validator:')
print(f'  оставили (integer в диапазоне):  {n_kept_pd}')
print(f'  сбросили в нет (out of range):    {n_dropped_pd}')
print(f'  без сигнала (нет pc или disc):    {n_no_signal}')

price_default validator:
  оставили (integer в диапазоне):  106
  сбросили в нет (out of range):    70
  без сигнала (нет pc или disc):    31


## Eval — STRICT vs FUZZY (для product_name)

In [18]:
# Post-processing fixes (from agent recommendations):
# 1. QR mirror: in GT, price1_qr==price_default (88%), price4_qr==price_card (95%), qr_code_barcode==barcode (99%)
# 2. Code dedup: if a code value dominates >40% of tracks in a video, it's a VLM hallucination → 'нет'
# 3. additional_info: snap to closed vocabulary

from collections import Counter
from rapidfuzz import process, fuzz

# --- 1. QR mirror ---
# GT convention: price_X uses comma ('252,63'), price_X_qr uses dot ('252.63')
# barcode/qr_code_barcode same format (both strings)
def to_dot(s):
    if s is None: return s
    return str(s).replace(',', '.')

n_qr_mirror = Counter()
for r in records:
    if r.get('price_default') not in (None, '', 'нет'):
        r['price1_qr'] = to_dot(r['price_default'])
        n_qr_mirror['price1_qr'] += 1
    if r.get('price_card') not in (None, '', 'нет'):
        r['price4_qr'] = to_dot(r['price_card'])
        n_qr_mirror['price4_qr'] += 1
    if r.get('barcode') not in (None, '', 'нет'):
        r['qr_code_barcode'] = r['barcode']
        n_qr_mirror['qr_code_barcode'] += 1

print(f'QR mirror filled: {dict(n_qr_mirror)}')

# --- 2. Per-video code hallucination suppression ---
by_video = {}
for r in records:
    by_video.setdefault(r['video'], []).append(r)

n_suppressed = 0
for vid, vid_records in by_video.items():
    non_net = [r for r in vid_records if r.get('code', 'нет') != 'нет']
    if len(non_net) < 10: continue  # skip small videos — 1/2 ratio gives false alarm
    code_counts = Counter(r['code'] for r in non_net)
    total_non_net = len(non_net)
    for code_val, cnt in code_counts.items():
        if cnt / total_non_net > 0.40:
            for r in non_net:
                if r['code'] == code_val:
                    r['code'] = 'нет'
                    n_suppressed += 1
            print(f'  Suppressed dominant code {code_val!r} in {vid}: {cnt}/{total_non_net} = {cnt/total_non_net:.0%}')

print(f'Code suppression: {n_suppressed} entries cleared')

# --- 3. additional_info: snap to closed vocabulary ---
_AI_VOCAB = [
    'Сухое', 'Полусухое', 'Полусладкое', 'Сладкое',
    '2 по цене 1 от цены без карты',
    '3 по цене 2 от цены без карты',
    'Полусухое, Упс! Товар закончился. Уже везём!',
    'Сухое, 3 по цене 2 от цены без карты',
    'Сухое, доп. скидка для участников соц. программы',
]

def snap_additional_info(raw):
    if not raw or str(raw).strip().lower() in ('нет', '', 'none'):
        return 'нет'
    raw_norm = str(raw).strip()
    for v in _AI_VOCAB:
        if raw_norm in v or v.startswith(raw_norm):
            return v
    result = process.extractOne(raw_norm, _AI_VOCAB, scorer=fuzz.token_set_ratio, score_cutoff=70)
    return result[0] if result else 'нет'

n_snapped = 0
for r in records:
    old = r.get('additional_info', 'нет')
    new = snap_additional_info(old)
    if new != old:
        r['additional_info'] = new
        n_snapped += 1

print(f'additional_info snapped: {n_snapped} entries adjusted')

# --- 4. product_name canonicalization (strip bleed-in + script fold) ---
import re as _re
_BLEED_TOKENS = _re.compile(
    r'(\s+\d+\s+по\s+цене\s+\d+.*|'
    r'\s+(Сухое|Полусухое|Полусладкое|Сладкое)(\W.*)?$)',
    _re.IGNORECASE
)
_VOLUME_ANCHOR = _re.compile(r'(\d[\.,]\s*\d{1,2}\s*[Lл])', _re.IGNORECASE)

def canonical_product_name(raw):
    if not raw or str(raw).strip() in ('', 'нет'): return 'нет'
    s = str(raw).strip()
    # Cut at volume anchor (everything after is usually bleed-in)
    m = _VOLUME_ANCHOR.search(s)
    if m:
        s = s[:m.end()].rstrip()
    # Strip bleed-in patterns
    s = _BLEED_TOKENS.sub('', s)
    s = _re.sub(r'\s+', ' ', s).strip()
    # GT-format space normalization: '0.75L' -> '0. 75L' (GT has 61 such, vs only 10 in 0.75L form)
    s = _re.sub(r'(\d)\.(\d{2}[LМмл])', r'\1. \2', s)
    return s if s else 'нет'

n_pn = 0
for r in records:
    old_pn = r.get('product_name', 'нет')
    new_pn = canonical_product_name(old_pn)
    if new_pn != old_pn:
        r['product_name'] = new_pn
        n_pn += 1
print(f'product_name canonicalized: {n_pn} entries cleaned')


QR mirror filled: {'price1_qr': 137, 'price4_qr': 360, 'qr_code_barcode': 1}
Code suppression: 0 entries cleared
additional_info snapped: 14 entries adjusted
product_name canonicalized: 60 entries cleaned


In [19]:
def iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1,bx1), max(ay1,by1)
    ix2, iy2 = min(ax2,bx2), min(ay2,by2)
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    if inter == 0: return 0.0
    return inter / ((ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - inter)

fps_map = {'25_12-20': 19.6, '26_12-20': 20.0, '43_15': 19.9}
labeled = set(fps_map)
lookup = {(r['video'], r['track_id']): r for r in records}

matches = []
for g in gt:
    vid = g['video']
    if vid not in labeled: continue
    gt_frame = int(round(g['frame_timestamp_ms'] / 1000.0 * fps_map[vid]))
    fm = per_frame.get(vid, {})
    best_iou, best_tid = 0.0, None
    for delta in range(-2, 3):
        for t in fm.get(str(gt_frame+delta), []):
            i = iou(g['bbox'], t['bbox'])
            if i > best_iou: best_iou, best_tid = i, t['track_id']
    if best_iou >= 0.5 and (vid, best_tid) in lookup:
        matches.append((g, lookup[(vid, best_tid)]))

print(f'Matched: {len(matches)}/{len(gt)}')

PER_TAG_THRESHOLD = 0.80
EVAL_FIELDS = [f for f in CSV_FIELDS if f not in ('video','track_id')]
BASE_CORRECT = 6
TOTAL_FIELDS = len(EVAL_FIELDS) + BASE_CORRECT

def norm(v):
    if v is None: return 'нет'
    s = str(v).strip()
    return s if s else 'нет'

def field_match_strict(field, gt_v, pr_v):
    return gt_v == pr_v

def field_match_fuzzy(field, gt_v, pr_v):
    # product_name — fuzzy через rapidfuzz; остальное strict
    if field == 'product_name':
        return product_name_match(pr_v, gt_v, threshold=80)
    return gt_v == pr_v

def evaluate(mode_fn, label):
    pred_by_id = {id(g): p for g, p in matches}
    tag_scores = []
    field_stats = {f: {'correct':0,'total':0} for f in EVAL_FIELDS}
    for g in gt:
        vid = g['video']
        if vid not in labeled: continue
        matched = id(g) in pred_by_id
        if matched:
            p = pred_by_id[id(g)]
            n_correct = BASE_CORRECT
            for f in EVAL_FIELDS:
                gt_v = norm(g.get(f, 'нет'))
                pr_v = norm(p.get(f, 'нет'))
                field_stats[f]['total'] += 1
                if mode_fn(f, gt_v, pr_v):
                    n_correct += 1
                    field_stats[f]['correct'] += 1
        else:
            n_correct = 0
        tag_scores.append({'video': vid, 'pct': n_correct / TOTAL_FIELDS, 'matched': matched})
    passed = sum(1 for t in tag_scores if t['pct'] >= PER_TAG_THRESHOLD)
    print(f'\n=== {label} ===')
    print(f'  Main metric ≥80%: {passed}/{len(tag_scores)} = {passed/len(tag_scores):.1%}')
    # bucket
    buckets = [(0.0,0.01),(0.5,0.7),(0.7,0.8),(0.8,0.9),(0.9,1.01)]
    for lo, hi in buckets:
        n = sum(1 for t in tag_scores if lo <= t['pct'] < hi)
        lab = '0% (миссы)' if lo==0 else f'{lo:.0%}-{hi:.0%}'
        print(f'    {lab:>13}: {n}')
    # per-field
    print(f'  Per-field (top wins / losses vs baseline):')
    rows = sorted(((f, s['correct'], s['total']) for f, s in field_stats.items() if s['total']>0),
                  key=lambda r: r[1]/r[2] if r[2] else 0, reverse=True)
    for f, c, t in rows:
        if c/t < 0.99:   # хайдим скучные 100% (always-empty)
            print(f'    {f:<22} {c}/{t} = {c/t:.1%}')
    return passed, field_stats

strict_passed, strict_stats = evaluate(field_match_strict, 'STRICT (как baseline)')
fuzzy_passed, fuzzy_stats   = evaluate(field_match_fuzzy,  'FUZZY (как у организаторов)')

Matched: 124/157

=== STRICT (как baseline) ===
  Main metric ≥80%: 3/157 = 1.9%
       0% (миссы): 33
          50%-70%: 97
          70%-80%: 23
          80%-90%: 3
         90%-101%: 0
  Per-field (top wins / losses vs baseline):
    discount_amount        84/124 = 67.7%
    price4_qr              82/124 = 66.1%
    price_card             79/124 = 63.7%
    additional_info        77/124 = 62.1%
    special_symbols        48/124 = 38.7%
    print_datetime         31/124 = 25.0%
    code                   31/124 = 25.0%
    id_sku                 8/124 = 6.5%
    price_default          7/124 = 5.6%
    price2_qr              7/124 = 5.6%
    price1_qr              6/124 = 4.8%
    barcode                5/124 = 4.0%
    qr_code_barcode        4/124 = 3.2%
    product_name           2/124 = 1.6%

=== FUZZY (как у организаторов) ===
  Main metric ≥80%: 4/157 = 2.5%
       0% (миссы): 33
          50%-70%: 85
          70%-80%: 34
          80%-90%: 4
         90%-101%: 0
  Per-field (t

## Submission — пишем `data/submission_v8.csv`

Финальный CSV в формате организаторов: 29 колонок, цены без QR в dot-decimal (`129.99`), отсутствующие значения как `'нет'`.

In [20]:
import pandas as pd

VIDEO_MAP = {
    '25_12-20': '25_12-20.mp4', '26_12-20': '26_12-20.mp4', '43_15': '43_15.mp4',
    'Unlabeled_25_12-20': '25_12-20.mp4', 'Unlabeled_26_12-20': '26_12-20.mp4',
    'Unlabeled_26_2-10': '26_2-10.mp4',
}

SUBMISSION_FIELDS = [
    'filename', 'product_name', 'price_default', 'price_card', 'price_discount',
    'barcode', 'discount_amount', 'id_sku', 'print_datetime', 'code',
    'additional_info', 'color', 'special_symbols',
    'frame_timestamp', 'x_min', 'y_min', 'x_max', 'y_max',
    'qr_code_barcode', 'price1_qr', 'price2_qr', 'price3_qr', 'price4_qr',
    'wholesale_level_1_count', 'wholesale_level_1_price',
    'wholesale_level_2_count', 'wholesale_level_2_price',
    'action_price_qr', 'action_code_qr',
]
COMMA_TO_DOT = {'price_default', 'price_card', 'price_discount'}

def comma_to_dot(v):
    if v in (None, '', 'нет'): return v
    s = str(v)
    if re.fullmatch(r'-?\d+,\d+', s): return s.replace(',', '.')
    return s

meta_lookup = {(m['video'], m['track_id']): m for m in meta_all}
rows = []
for r in records:
    m = meta_lookup.get((r['video'], r['track_id']))
    bbox = m['bbox'] if m else [0,0,0,0]
    ts = int(round(m['best_timestamp_ms'])) if m else 0
    row = {
        'filename': VIDEO_MAP[r['video']],
        'frame_timestamp': ts,
        'x_min': round(float(bbox[0]), 1),
        'y_min': round(float(bbox[1]), 1),
        'x_max': round(float(bbox[2]), 1),
        'y_max': round(float(bbox[3]), 1),
    }
    for f in SUBMISSION_FIELDS:
        if f in row: continue
        v = r.get(f, 'нет')
        if f in COMMA_TO_DOT: v = comma_to_dot(v)
        row[f] = v
    rows.append(row)

SUB_OUT = ROOT / 'data' / 'submission_v8.csv'
df = pd.DataFrame(rows, columns=SUBMISSION_FIELDS)
df.to_csv(SUB_OUT, sep=',', index=False, encoding='utf-8')
print(f'Written: {SUB_OUT} ({SUB_OUT.stat().st_size//1024} KB, {len(df)} rows)')

# Eval report
EVAL_OUT = ROOT / 'data' / 'eval_report_v8.json'
EVAL_OUT.write_text(json.dumps({
    'strict_passed': strict_passed,
    'fuzzy_passed':  fuzzy_passed,
    'total':         len(matches) + sum(1 for g in gt if g['video'] in labeled and id(g) not in {id(x[0]) for x in matches}),
    'matched':       len(matches),
    'strict_per_field': {f: s['correct']/max(s['total'],1) for f, s in strict_stats.items()},
    'fuzzy_per_field':  {f: s['correct']/max(s['total'],1) for f, s in fuzzy_stats.items()},
}, ensure_ascii=False, indent=2))
print(f'Eval saved: {EVAL_OUT}')

Written: /Users/alex/ML Projects/Lenta Tech Life Hack/solution/data/submission_v8.csv (120 KB, 544 rows)
Eval saved: /Users/alex/ML Projects/Lenta Tech Life Hack/solution/data/eval_report_v8.json
